# 🔢 SirenNet GRU → TFLite INT8

**Pipeline**: PyTorch `.pth` → ONNX → TensorFlow SavedModel → TFLite INT8

**Yêu cầu**:
- `siren_gru_best.pth` đã có trên Drive (`MyDrive/siren_gru_output/`)
- Dataset zip vẫn ở `MyDrive/dataset.zip` (dùng làm calibration data cho quantize)
- Runtime → **T4 GPU** (hoặc CPU cũng được, pipeline này không nặng)

**Output**: `siren_gru_int8.tflite` — đưa thẳng vào STM32CubeAI Studio

## 0. Cài thư viện

In [ ]:
!pip install -q onnx onnxsim onnx2tf tensorflow
!pip install -q 'onnx2tf>=1.20.0'
import tensorflow as tf
import torch
print(f'TF  : {tf.__version__}')
print(f'PyTorch: {torch.__version__}')


## 1. Mount Drive & Hyperparameters

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch

WEIGHT_PATH = '/content/drive/MyDrive/siren_gru_output/siren_gru_best.pth'
DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'
OUT_DIR     = '/content/drive/MyDrive/siren_gru_output'

SR         = 22050
N_MELS     = 64
N_MFCC     = 40
HOP_LEN    = 512
N_FFT      = 1024
FMIN       = 300
FMAX       = 3500
CLIP_SEC   = 2.0
CLIP_LEN   = int(CLIP_SEC * SR)
STRIDE_SEC = 0.17
PRE_EMPH   = 0.97
MEL_FREQ_AFTER_POOL  = N_MELS // 4
MFCC_FREQ_AFTER_POOL = N_MFCC // 4
N_FRAMES   = CLIP_LEN // HOP_LEN + 1
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Input mel : (1, 1, {N_MELS}, {N_FRAMES})')
print(f'Input mfcc: (1, 1, {N_MFCC}, {N_FRAMES})')


## 2. Định nghĩa Model (ManualGRU — copy từ notebook train)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNNBranch(nn.Module):
    def __init__(self, in_ch=1, base=32, freq_after_pool=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base,    3, padding=1, bias=False), nn.BatchNorm2d(base),   nn.GELU(),
            nn.Conv2d(base,   base,   3, padding=1, bias=False), nn.BatchNorm2d(base),   nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(base,   base*2, 3, padding=1, bias=False), nn.BatchNorm2d(base*2), nn.GELU(),
            nn.Conv2d(base*2, base*2, 3, padding=1, bias=False), nn.BatchNorm2d(base*2), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(base*2, base*4, 3, padding=1, bias=False), nn.BatchNorm2d(base*4), nn.GELU(),
            nn.Conv2d(base*4, base*4, 3, padding=1, bias=False), nn.BatchNorm2d(base*4), nn.GELU(),
            nn.Dropout2d(0.2),
        )
        self.freq_pool = nn.AvgPool2d(kernel_size=(freq_after_pool, 1))
    def forward(self, x):
        h = self.net(x)
        h = self.freq_pool(h)
        return h.squeeze(2).permute(0, 2, 1)

class ManualGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.H  = hidden_size
        self.ih = nn.Linear(input_size,  hidden_size * 3)
        self.hh = nn.Linear(hidden_size, hidden_size * 3)
    def forward(self, x, h):
        gi = self.ih(x); gh = self.hh(h); H = self.H
        r = torch.sigmoid(gi[:, :H]    + gh[:, :H])
        z = torch.sigmoid(gi[:, H:2*H] + gh[:, H:2*H])
        n = torch.tanh(   gi[:, 2*H:]  + r * gh[:, 2*H:])
        return (1 - z) * n + z * h

class ManualGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.3):
        super().__init__()
        self.H = hidden_size; self.L = num_layers; self.dropout_p = dropout
        self.cells = nn.ModuleList([
            ManualGRUCell(input_size if i == 0 else hidden_size, hidden_size)
            for i in range(num_layers)
        ])
    def forward(self, x):
        B, T, _ = x.shape
        h = [torch.zeros(B, self.H, device=x.device, dtype=x.dtype) for _ in range(self.L)]
        outputs = []
        for t in range(T):
            inp = x[:, t, :]
            for i, cell in enumerate(self.cells):
                h[i] = cell(inp, h[i]); inp = h[i]
                if self.training and self.dropout_p > 0 and i < self.L - 1:
                    inp = F.dropout(inp, p=self.dropout_p)
            outputs.append(inp)
        return torch.stack(outputs, dim=1)

class SirenNetV4_GRU(nn.Module):
    def __init__(self, n_classes=2, base_ch=48, gru_hidden=512, gru_layers=2):
        super().__init__()
        self.mel_cnn  = CNNBranch(1, base_ch,      MEL_FREQ_AFTER_POOL)
        self.mfcc_cnn = CNNBranch(1, base_ch // 2, MFCC_FREQ_AFTER_POOL)
        total_ch = base_ch * 4 + (base_ch // 2) * 4
        self.gru = ManualGRU(total_ch, gru_hidden, gru_layers, dropout=0.3)
        self.attn_w = nn.Linear(gru_hidden, 1)
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden, 512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 128),        nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, mel, mfcc):
        mel_f  = self.mel_cnn(mel)
        mfcc_f = self.mfcc_cnn(mfcc)
        x = torch.cat([mel_f, mfcc_f], dim=-1)
        gru_out = self.gru(x)
        a   = torch.softmax(self.attn_w(gru_out), dim=1)
        ctx = (gru_out * a).sum(dim=1)
        return self.classifier(ctx)

model = SirenNetV4_GRU().to(DEVICE)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
model.eval()
print('Model loaded OK')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')


## 3. Export ONNX (FP32)

In [ ]:
import onnx

ONNX_PATH = '/content/siren_gru_fp32.onnx'
model.eval().cpu()
d_mel  = torch.randn(1, 1, N_MELS,  N_FRAMES)
d_mfcc = torch.randn(1, 1, N_MFCC, N_FRAMES)

with torch.no_grad():
    torch.onnx.export(
        model, (d_mel, d_mfcc), ONNX_PATH,
        input_names=['mel', 'mfcc'],
        output_names=['logits'],
        opset_version=18,
    )

m = onnx.load(ONNX_PATH)
onnx.checker.check_model(m)
size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f'ONNX FP32: {size_mb:.2f} MB')


## 4. ONNX → TensorFlow SavedModel

In [ ]:
!onnx2tf \
    -i /content/siren_gru_fp32.onnx \
    -o /content/saved_model \
    -osd \
    --non_verbose

import tensorflow as tf
# Kiểm tra SavedModel load được
loaded = tf.saved_model.load('/content/saved_model')
print('SavedModel loaded OK')


## 5. Chuẩn bị Calibration Data

INT8 quantize cần **representative dataset** — dùng ~200 samples từ dataset thật
để calibrate range của activation. Nếu dùng data random thì quantize kém chính xác.


In [ ]:
import zipfile, numpy as np, librosa
from pathlib import Path
from scipy.signal import butter, sosfilt

OLD_DIR = '/content/dataset'
if not os.path.exists(OLD_DIR):
    print('Giải nén...')
    with zipfile.ZipFile(DATASET_ZIP) as z: z.extractall('/content')

def pre_emphasis(y, c=0.97): return np.append(y[0], y[1:] - c * y[:-1])
def bandpass(y, sr):
    sos = butter(4, [500, 1800], btype='band', fs=sr, output='sos')
    return sosfilt(sos, y)
def normalize(x): return (x - x.mean()) / (x.std() + 1e-6)
def fix_len(x, T): return x[:, :T] if x.shape[1] >= T else np.pad(x, ((0,0),(0,T-x.shape[1])))

def wav_to_inputs(wav_path):
    y, sr = librosa.load(wav_path, sr=SR, mono=True)
    y = bandpass(y, sr)
    if len(y) < CLIP_LEN:
        pad = np.zeros(CLIP_LEN); pad[:len(y)] = y; y = pad
    seg = y[:CLIP_LEN]
    mel = normalize(librosa.power_to_db(
        librosa.feature.melspectrogram(y=pre_emphasis(seg), sr=SR,
            n_fft=N_FFT, hop_length=HOP_LEN, n_mels=N_MELS, fmin=FMIN, fmax=FMAX),
        ref=np.max)).astype(np.float32)
    mfcc = normalize(librosa.feature.mfcc(y=pre_emphasis(seg), sr=SR,
        n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LEN, fmin=FMIN, fmax=FMAX
    ).astype(np.float32))
    return fix_len(mel, N_FRAMES)[np.newaxis, np.newaxis], fix_len(mfcc, N_FRAMES)[np.newaxis, np.newaxis]

# Lấy tối đa 100 file mỗi class
import random
random.seed(42)
wav_files = []
for folder in ['positive', 'negative']:
    fp = Path(OLD_DIR) / folder
    if fp.exists():
        wavs = list(fp.glob('*.wav')) + list(fp.glob('*.WAV'))
        wav_files += random.sample(wavs, min(100, len(wavs)))

print(f'Calibration files: {len(wav_files)}')

# Extract features
calib_mel, calib_mfcc = [], []
errors = 0
for wp in wav_files:
    try:
        m, mc = wav_to_inputs(str(wp))
        calib_mel.append(m); calib_mfcc.append(mc)
    except: errors += 1

calib_mel  = np.concatenate(calib_mel,  axis=0).astype(np.float32)
calib_mfcc = np.concatenate(calib_mfcc, axis=0).astype(np.float32)
print(f'Calib mel : {calib_mel.shape}')
print(f'Calib mfcc: {calib_mfcc.shape}')
if errors: print(f'Errors: {errors}')


## 6. Quantize INT8 → TFLite

Dùng **Full Integer Quantization** — cả weight lẫn activation đều INT8.
Đây là mode ST Edge AI hỗ trợ tốt nhất và cho Flash nhỏ nhất.


In [ ]:
import tensorflow as tf
import numpy as np

def representative_dataset():
    for i in range(min(len(calib_mel), 100)):
        # Đảm bảo shape là (1, 1, 64, 87) và (1, 1, 40, 87) kiểu float32
        mel_input = calib_mel[i].astype(np.float32)
        mfcc_input = calib_mfcc[i].astype(np.float32)
        
        if mel_input.ndim == 3: # (1, 64, 87)
             mel_input = np.expand_dims(mel_input, axis=0)
        if mfcc_input.ndim == 3:
             mfcc_input = np.expand_dims(mfcc_input, axis=0)

        yield [mel_input, mfcc_input]

# Tải SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model('/content/saved_model')

# CẤU HÌNH QUAN TRỌNG:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

converter.target_spec.supported_types = [tf.int8]

# converter.target_spec.supported_ops += [tf.lite.OpsSet.SELECT_TF_OPS]

try:
    print("Bắt đầu chuyển đổi model sang INT8...")
    tflite_int8 = converter.convert()
    with open('/content/siren_gru_int8.tflite', 'wb') as f:
        f.write(tflite_int8)
    print("✅ Chuyển đổi thành công!")
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("\nKiểm tra lại: Có thể trục Time (87) trong model đang là 'None'.")
    print("Hãy thử resize tất cả calib_data về đúng kích thước (64, 87) trước khi yield.")

## 7. Kiểm tra Accuracy sau Quantize

In [ ]:
# So sánh output FP32 vs INT8 trên calibration data
import numpy as np

# Load 2 interpreter
interp_fp32 = tf.lite.Interpreter(model_path=TFLITE_FP32_PATH)
interp_int8 = tf.lite.Interpreter(model_path=TFLITE_INT8_PATH)
interp_fp32.allocate_tensors()
interp_int8.allocate_tensors()

in_fp32  = interp_fp32.get_input_details()
out_fp32 = interp_fp32.get_output_details()
in_int8  = interp_int8.get_input_details()
out_int8 = interp_int8.get_output_details()

print('INT8 input  dtype:', in_int8[0]['dtype'],  '| scale:', in_int8[0]['quantization'])
print('INT8 output dtype:', out_int8[0]['dtype'], '| scale:', out_int8[0]['quantization'])

# Chạy 50 samples đầu
N_CHECK = min(50, len(calib_mel))
agree = 0
for i in range(N_CHECK):
    mel_i  = calib_mel[i:i+1]
    mfcc_i = calib_mfcc[i:i+1]

    # FP32
    interp_fp32.set_tensor(in_fp32[0]['index'], mel_i)
    interp_fp32.set_tensor(in_fp32[1]['index'], mfcc_i)
    interp_fp32.invoke()
    pred_fp32 = np.argmax(interp_fp32.get_tensor(out_fp32[0]['index']))

    # INT8 — cần scale input
    scale, zp = in_int8[0]['quantization']
    mel_q  = (mel_i  / scale + zp).astype(np.int8)
    scale2, zp2 = in_int8[1]['quantization']
    mfcc_q = (mfcc_i / scale2 + zp2).astype(np.int8)
    interp_int8.set_tensor(in_int8[0]['index'], mel_q)
    interp_int8.set_tensor(in_int8[1]['index'], mfcc_q)
    interp_int8.invoke()
    pred_int8 = np.argmax(interp_int8.get_tensor(out_int8[0]['index']))

    if pred_fp32 == pred_int8: agree += 1

print(f'\nFP32 vs INT8 agreement: {agree}/{N_CHECK} = {agree/N_CHECK*100:.1f}%')
print('(>95% là quantize tốt, >98% là xuất sắc)')


## 8. Lưu về Drive & Báo cáo kết quả

In [ ]:
import shutil

dst_int8 = OUT_DIR + '/siren_gru_int8.tflite'
dst_fp32 = OUT_DIR + '/siren_gru_fp32.tflite'
shutil.copy(TFLITE_INT8_PATH, dst_int8)
shutil.copy(TFLITE_FP32_PATH, dst_fp32)

onnx_kb = os.path.getsize(ONNX_PATH) / 1024

print('=' * 50)
print('KẾT QUẢ QUANTIZE')
print('=' * 50)
print(f'ONNX FP32    : {onnx_kb:>8.1f} KB  ({onnx_kb/1024:.2f} MB)')
print(f'TFLite FP32  : {fp32_kb:>8.1f} KB  ({fp32_kb/1024:.2f} MB)')
print(f'TFLite INT8  : {int8_kb:>8.1f} KB  ({int8_kb/1024:.2f} MB)')
print(f'Giảm so với ONNX gốc: {onnx_kb/int8_kb:.1f}x')
print()
print('CHIP PHÙ HỢP (ước tính RAM activation ~25% Flash):')
ram_est = int8_kb * 0.25
if int8_kb < 512 and ram_est < 128:
    print('  -> STM32F411E  (512KB Flash, 128KB RAM) -- FIT!')
elif int8_kb < 1024 and ram_est < 256:
    print('  -> STM32F429ZI (2MB Flash,  256KB RAM)  -- FIT')
elif int8_kb < 2048 and ram_est < 512:
    print('  -> STM32H743ZI (2MB Flash,  1MB RAM)    -- FIT')
else:
    print('  -> Cần thu nhỏ kiến trúc thêm (KD hoặc student nhỏ hơn)')
print()
print(f'File đưa vào STM32CubeAI Studio: {dst_int8}')
